# Домашняя работа 2 — качество данных

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal

DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)

print("pandas", pd.__version__)

## Независимая загрузка

In [ ]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)

print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

## H1. Собственная функция очистки

In [ ]:
def clean_export(source):
    # Копия входа, чтобы не менять исходный объект
    df = source.copy(deep=True)
    original_cols = list(source.columns)

    audit = {
        'input': len(df),
        'replays': 0,
        'clean': 0,
        'quarantine': 0,
        'parse_errors': 0,
    }

    # Технический ключ для поиска одинаковых source_row
    df['_row_key'] = pd.to_numeric(df['source_row'], errors='coerce')

    # Проверка конфликтующих версий и удаление повторных доставок
    drop_idx = []
    known = df['_row_key'].notna()
    for key, group in df.loc[known].groupby('_row_key', sort=False):
        if len(group) > 1:
            unique = group.drop_duplicates(subset=original_cols)
            assert len(unique) == 1, f'Конфликт версий для source_row {key}'
            drop_idx.extend(group.index[1:])

    dedup = df.drop(index=drop_idx).drop(columns=['_row_key'])
    audit['replays'] = len(drop_idx)

    # Исходные значения для карантина и подсчёта ошибок формата
    row_raw = dedup['source_row']
    action_raw = dedup['action_taken']
    loan_raw = dedup['loan_amount']
    income_raw = dedup['income']
    county_raw = dedup['county_code']

    row_num = pd.to_numeric(row_raw, errors='coerce')
    action_num = pd.to_numeric(action_raw, errors='coerce')
    loan_num = pd.to_numeric(loan_raw, errors='coerce')
    income_num = pd.to_numeric(income_raw, errors='coerce')
    county_norm = county_raw.str.strip()

    # Значение было, но числом не стало
    parse_errors = 0
    for raw, num in (
        (row_raw, row_num),
        (action_raw, action_num),
        (loan_raw, loan_num),
        (income_raw, income_num),
    ):
        parse_errors += int((raw.notna() & num.isna()).sum())
    audit['parse_errors'] = parse_errors

    # Проверка обязательных правил
    reasons = []
    for idx in dedup.index:
        r = []
        if pd.isna(row_num.loc[idx]):
            r.append('некорректный source_row')
        if pd.isna(action_num.loc[idx]) or action_num.loc[idx] not in (1, 2, 3):
            r.append('некорректный action_taken')
        if pd.isna(loan_num.loc[idx]) or loan_num.loc[idx] <= 0:
            r.append('некорректная сумма кредита')
        county = county_norm.loc[idx]
        if pd.isna(county) or county != '11001':
            r.append('некорректная территория')
        if pd.notna(income_raw.loc[idx]) and pd.isna(income_num.loc[idx]):
            r.append('некорректный доход')
        reasons.append(';'.join(r))

    reasons = pd.Series(reasons, index=dedup.index, dtype='object')
    clean_mask = reasons == ''

    # Подготовленный слой с числовыми типами
    clean = dedup.loc[clean_mask, original_cols].copy()
    clean['source_row'] = row_num[clean_mask].astype('Int64')
    clean['action_taken'] = action_num[clean_mask].astype('Int64')
    clean['loan_amount'] = loan_num[clean_mask].astype(float)
    clean['income'] = income_num[clean_mask].astype(float)
    clean['county_code'] = county_norm[clean_mask]

    # Карантин с исходными значениями
    quarantine = dedup.loc[~clean_mask, original_cols].copy()
    quarantine['reason'] = reasons[~clean_mask]

    audit['clean'] = len(clean)
    audit['quarantine'] = len(quarantine)

    assert audit['input'] == audit['replays'] + audit['clean'] + audit['quarantine']
    return clean, quarantine, audit

## H2. Год и известный доход

In [ ]:
clean, quarantine, audit = clean_export(export)
print(f'Журнал: {audit}')

year_ok = clean['activity_year'].eq('2023').fillna(False).all()
print(f'Год везде 2023: {year_ok}')

n_total = len(clean)
known_income = int(clean['income'].notna().sum())
unknown_income = int(clean['income'].isna().sum())
mean_income = clean['income'].mean()

print(f'Всего записей: {n_total}')
print(f'Известный доход: {known_income}')
print(f'Неизвестный доход: {unknown_income}')
print(f'Средний доход: {mean_income} тысяч USD')
print('Единица измерения: тысячи USD')

balance_ok = audit['input'] == audit['replays'] + audit['clean'] + audit['quarantine']
print(f'Баланс сходится: {balance_ok}')

assert year_ok
assert balance_ok
assert_frame_equal(export, raw_copy)
print('Вход не изменился')

## H3. Проверка отчётного года

In [ ]:
# Меняем год первой строки только в копии export
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], 'activity_year'] = '2022'

year_ok_wrong = wrong_year['activity_year'].eq('2023').fillna(False).all()
print(f'Проверка года на испорченных данных: {year_ok_wrong}')
assert not year_ok_wrong

assert_frame_equal(export, raw_copy)
print('export сохранён')

print('Примеры известных пропусков дохода:')
print(clean.loc[
    clean['income'].isna(),
    ['source_row', 'loan_amount', 'income']
].head())

print('Примеры ошибок формата дохода:')
print(quarantine.loc[
    quarantine['reason'].str.contains('некорректный доход', na=False),
    ['source_row', 'loan_amount', 'income', 'reason']
].head())

print('Примеры ошибок формата суммы:')
print(quarantine.loc[
    quarantine['reason'].str.contains('некорректная сумма кредита', na=False),
    ['source_row', 'loan_amount', 'income', 'reason']
].head())

print('Примеры необычно больших сумм:')
print(clean.loc[
    clean['loan_amount'] > 5_000_000,
    ['source_row', 'loan_amount', 'income']
].head())

## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей